## Synthetic test generation from multi-lingual and cross-lingual corpus

In this notebook, you'll learn how to adapt synthetic test data generation to multi-lingual (non english) and cross-lingual settings. For the sake of this tutorial, I am generating queries in Spanish from Spanish wikipedia articles. 

### Download and Load corpus

In [1]:
! git clone https://huggingface.co/datasets/vibrantlabsai/Sample_non_english_corpus

Cloning into 'Sample_non_english_corpus'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 0), reused 0 (delta 0), pack-reused 4 (from 1)
Unpacking objects: 100% (12/12), 11.43 KiB | 780.00 KiB/s, done.


In [ ]:
from pathlib import Path

from ragas.testset.document import Document

path = "Sample_Docs_Markdown/"
docs = [
    Document(page_content=p.read_text(), metadata={"source": str(p)})
    for p in Path(path).rglob("*.md")
]

In [3]:
len(docs)

6

### Initialize required models

In [ ]:
from openai import OpenAI

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory

openai_client = OpenAI()
generator_llm = llm_factory("gpt-4o-mini", client=openai_client)
generator_embeddings = embedding_factory(
    "openai", model="text-embedding-3-small", client=openai_client
)

### Setup Persona and transforms
you may automatically create personas using this [notebook](./_persona_generator.md). For the sake of simplicity, I am using a pre-defined person, two basic transforms and simple specific query distribution.

In [16]:
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="curious student",
        role_description="A student who is curious about the world and wants to learn more about different cultures and languages",
    ),
]

In [ ]:
from ragas.testset.transforms.extractors.llm_based import NERExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter

transforms = [HeadlineSplitter(), NERExtractor()]

### Initialize test generator

In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings, persona_list=personas
)

### Load and Adapt Queries

Here we load the required query types and adapt them to the target language. 

In [19]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]

for query, _ in distribution:
    prompts = await query.adapt_prompts("spanish", llm=generator_llm)
    query.set_prompts(**prompts)

### Generate

In [ ]:
dataset = generator.generate_with_docs(
    docs[:],
    testset_size=5,
    transforms=transforms,
    query_distribution=distribution,
)

In [22]:
eval_dataset = dataset.to_evaluation_dataset()

In [26]:
print("Query:", eval_dataset[0].user_input)
print("Reference:", eval_dataset[0].reference)

Query: Quelles sont les caractéristiques du Bronx en tant que borough de New York?
Reference: Le Bronx est l'un des cinq arrondissements de New York, qui est la plus grande ville des États-Unis. Bien que le contexte ne fournisse pas de détails spécifiques sur le Bronx, il mentionne que New York est une ville cosmopolite avec de nombreux quartiers ethniques, ce qui pourrait inclure des caractéristiques culturelles variées présentes dans le Bronx.


That's it. You can customize the test generation process as per your requirements.